In [236]:
import numpy as np
import matplotlib.pyplot as plt

In [237]:
x = np.fromfile("demodulated_60s.f32", dtype=np.float32)

# Remove junk before signal stabilizes
start = np.argmin(np.abs(x) < np.mean(np.abs(x)))
x = x[start:]
# x = x / np.mean(np.abs(x))

x[x > 0] = 1
x[x < 0] = -1

In [238]:
# Get first transition
diff = np.diff(x)
offset = np.argmax(np.abs(diff))

# Sample at midpoint of each bit
x = x[offset::20]

In [239]:
preamble = "10001011"
preamble = [1 if c == '1' else -1 for c in preamble]
preamble = np.array(preamble)

In [240]:
def parity(word, last):
    # word = 30 bits, -1 or 1
    # last = [D29*, D30*], -1 or 1

    word = np.array(list(word))

    if last[1] != 1:
        word[0:24] = word[0:24] * -1

    parity_idx = [
        [1, 2, 3, 5, 6, 10, 11, 12, 13, 14, 17, 18, 20, 23],
        [2, 3, 4, 6, 7, 11, 12, 13, 14, 15, 18, 19, 21, 24],
        [1, 3, 4, 5, 7, 8, 12, 13, 14, 15, 16, 19, 20, 22],
        [2, 4, 5, 6, 8, 9, 13, 14, 15, 16, 17, 20, 21, 23],
        [1, 3, 5, 6, 7, 9, 10, 14, 15, 16, 17, 18, 21, 22, 24],
        [3, 5, 6, 8, 9, 10, 11, 13, 15, 19, 22, 23, 24]
    ]

    # Shift because 0 indexed
    parity_idx = [[b - 1 for b in a] for a in parity_idx]

    d25 = last[0] * np.prod(word[parity_idx[0]])
    d26 = last[1] * np.prod(word[parity_idx[1]])
    d27 = last[0] * np.prod(word[parity_idx[2]])
    d28 = last[1] * np.prod(word[parity_idx[3]])
    d29 = last[1] * np.prod(word[parity_idx[4]])
    d30 = last[0] * np.prod(word[parity_idx[5]])

    valid = np.sum([
        word[24] == d25,
        word[25] == d26,
        word[26] == d27,
        word[27] == d28,
        word[28] == d29,
        word[29] == d30
    ])

    if valid == 6:
        return int(-1 * last[1])
    else:
        return 0

In [241]:
def arr2bin(arr, t=1):
    return "".join(['1' if c == t else '0' for c in arr])

In [242]:
# Find preambles

i = 0

valid_preambles = []

while True:
    sample = x[i:i+8]

    has_preamble = False
    inverted = False

    if np.equal(sample, preamble).all():
        has_preamble = True
    elif np.equal(sample, -preamble).all():
        has_preamble = True
        inverted = True

    if has_preamble:
        word = x[i:i+30]

        if inverted:
            word = -word

        valid = parity(word, [-1, -1])

        if valid:
            valid_preambles.append(i)

            if inverted:
                x = -x
                print("Inverted")

    i = i + 1

    # Don't continue if less than 1 word is left
    if i > len(x) - 30:
        break

print(valid_preambles)
print(np.diff([a for a in valid_preambles]))

Inverted
[272, 572, 872, 1172, 1472, 1772, 2072, 2372, 2672]
[300 300 300 300 300 300 300 300]


In [243]:
# Find and validate subframes

subframes = []

last_parity = [-1, -1]

for i in valid_preambles:
    if len(x) >= i + 300:
        subframe = []

        for j in range(10):
            word = x[i + j*30 : i + (j+1)*30]
            word = np.array(list(word))

            valid = parity(word, last_parity)

            assert valid != 0, "Invalid word"

            subframe.append(arr2bin(word * valid))

            last_parity = word[-2:]
        
        subframes.append(subframe)

print(np.shape(subframes))

(8, 10)


In [317]:
# Decode subframes

def decode(sbf, word, start, end, twos_comp=False):
    # word, start, and end are 1 indexed to match GPS spec

    if start == end:
        return int(sbf[word-1][start-1])

    data = int(sbf[word-1][start-1:end], 2)
    data_len = end-start+1

    if twos_comp and data > 2**(data_len-1):
        data = data - 2**data_len

    return data

for sbf in subframes:
    tlm = sbf[0]
    how = sbf[1]

    # Word 2
    tow = decode(sbf, 2, 1, 17)
    sbf_id = decode(sbf, 2, 20, 22)

    print(f"Subframe ID: {sbf_id}, TOW: {tow}")

    if sbf_id == 1:
        week = decode(sbf, 3, 1, 10) # 20.3.3.3.1.1
        l2_code = decode(sbf, 3, 11, 12) # 20.3.3.3.1.2
        ura = decode(sbf, 3, 13, 16) # 20.3.3.3.1.3
        sv_health = decode(sbf, 3, 16, 16) # 20.3.3.3.1.4

        # 20.3.3.3.1.5
        iodc = decode(sbf, 3, 23, 24) << 8
        iodc = iodc | decode(sbf, 8, 1, 8)

        t_gd = decode(sbf, 7, 17, 24, True) * 2**-31 # 20.3.3.3.1.7

        # Clock correction - 20.3.3.3.1.8
        t_oc = decode(sbf, 8, 9, 24) * 2**4
        a_f2 = decode(sbf, 9, 1, 8, True) * 2**-55
        a_f1 = decode(sbf, 9, 9, 24, True) * 2**-43
        a_f0 = decode(sbf, 10, 1, 22, True) * 2**-31

        print(f"Week {week}, IODC {iodc}")
        print(f"t_oc={t_oc}, a_f2={a_f2:0.3e}, a_f1={a_f1:0.3e}, a_f0={a_f0:0.3e}")
    elif sbf_id == 2:
        iode = decode(sbf, 3, 1, 8)
        c_rs = decode(sbf, 3, 9, 24, True) * 2**-5
        delta_n = decode(sbf, 4, 1, 16, True) * 2**-43

        m_0 = decode(sbf, 4, 17, 24) << 24
        m_0 = m_0 | decode(sbf, 5, 1, 24)
        if m_0 > 2**31:
            m_0 = m_0 - 2**32
        m_0 = m_0 * 2**-31

        c_uc = decode(sbf, 6, 1, 16, True) * 2**-29

        e = decode(sbf, 6, 17, 24) << 24
        e = e | decode(sbf, 7, 1, 24)
        e = e * 2**-33
        assert e >= 0 and e <= 0.03, f"{e} is not valid"

        c_us = decode(sbf, 8, 1, 16, True) * 2**-29

        sqrt_a = decode(sbf, 8, 17, 24) << 24
        sqrt_a = sqrt_a | decode(sbf, 9, 1, 24)
        sqrt_a = sqrt_a * 2**-19
        assert sqrt_a >= 2530 and sqrt_a <= 8192, f"{sqrt_a} is not valid"

        t_oe = decode(sbf, 10, 1, 16) * 2**4
        assert t_oe < 604784, f"{t_oe} is not valid"

        fit_interval = decode(sbf, 10, 17, 17)
        aodo = decode(sbf, 10, 17, 22)

        print(f"iode={iode}, c_rs={c_rs:0.1f}, delta_n={delta_n:0.3e}, M0={m_0:0.3f}, c_uc={c_uc:0.3e}")
        print(f"e={e:0.4f}, c_us={c_us:0.3e}, sqrtA={sqrt_a:0.1f}, t_oe={t_oe}")
    elif sbf_id == 3:
        c_ic = decode(sbf, 3, 1, 16, True) * 2**-29

        omega_0 = decode(sbf, 3, 17, 24) << 24
        omega_0 = omega_0 | decode(sbf, 4, 1, 24)
        if omega_0 > 2**31:
            omega_0 = omega_0 - 2**32
        omega_0 = omega_0 * 2**-31

        c_is = decode(sbf, 5, 1, 24, True) * 2**-29

        i_0 = decode(sbf, 5, 17, 24, True) << 24
        i_0 = i_0 | decode(sbf, 6, 1, 24)
        if i_0 > 2**31:
            i_0 = i_0 - 2**32
        i_0 = i_0 * 2**-31

        c_rc = decode(sbf, 7, 1, 16, True) * 2**-5

        omega = decode(sbf, 7, 17, 24) << 24
        omega = omega | decode(sbf, 8, 1, 24)
        if omega > 2**31:
            omega = omega - 2**32
        omega = omega * 2**-31

        omega_dot = decode(sbf, 9, 1, 24, True) * 2**-43
        assert omega_dot >= -6.33e-7 and omega_dot <= 0, f"{omega_dot} is not valid"
        idot = decode(sbf, 10, 9, 22, True) * 2**-43

        iode = decode(sbf, 10, 1, 8)

        print(f"c_ic={c_ic:0.3e}, omega_0={omega_0:0.3f}, c_is={c_is:0.3e}, i_0={i_0:0.3f}, c_rc={c_rc:0.2f}")
        print(f"omega={omega:0.3f}, omega_dot={omega_dot:0.3e}, idot={idot:0.3e}, iode={iode}")

    print()

Subframe ID: 2, TOW: 57602
iode=53, c_rs=-1.8, delta_n=1.623e-09, M0=-0.146, c_uc=1.676e-08
e=0.0134, c_us=3.790e-06, sqrtA=5153.6, t_oe=345600

Subframe ID: 3, TOW: 57603
c_ic=-2.794e-08, omega_0=-0.520, c_is=4.966e-05, i_0=0.307, c_rc=306.03
omega=0.253, omega_dot=-2.733e-09, idot=-5.082e-11, iode=53

Subframe ID: 4, TOW: 57604

Subframe ID: 5, TOW: 57605

Subframe ID: 1, TOW: 57606
Week 231, IODC 53
t_oc=345600, a_f2=0.000e+00, a_f1=5.457e-12, a_f0=-4.546e-04

Subframe ID: 2, TOW: 57607
iode=53, c_rs=-1.8, delta_n=1.623e-09, M0=-0.146, c_uc=1.676e-08
e=0.0134, c_us=3.790e-06, sqrtA=5153.6, t_oe=345600

Subframe ID: 3, TOW: 57608
c_ic=-2.794e-08, omega_0=-0.520, c_is=4.966e-05, i_0=0.307, c_rc=306.03
omega=0.253, omega_dot=-2.733e-09, idot=-5.082e-11, iode=53

Subframe ID: 4, TOW: 57609

